Now we will construct our binary at-risk response variable based on if a patient started new blood pressure, diabetes, or new cholesterol/lipid medication after NHT start. We will also clean up and standardize other various columns. Features with over 50% null values have been removed.

In [233]:
import os
import pandas as pd
import numpy as np

In [234]:
PROCESSED_FILE = os.path.join("..", "data", "processed", "cardio_onc_prostate_03cleaned.csv")
CLEANED_FILE = os.path.join("..", "data", "processed", "cardio_onc_prostate_04cleaned.csv")

df = pd.read_csv(PROCESSED_FILE)

## Change bp_meds_post, lipid_meds_post, dm_meds_post to be binary 0/1
These are the columns we will use to construct our target variable. Right now all of them are (0: None; 1: One; 2: More than two). Changing to (0: None, 1: One or more). Keep NAs as is.

In [235]:
cols = ['bp_meds_post', 'lipid_meds_post', 'dm_meds_post']

print("Before:")
for col in cols:
    print(df[col].value_counts(dropna=False))

df[cols] = df[cols].where(df[cols].isna(), (df[cols] != 0).astype(int))

print("After:")
for col in cols:
    print(df[col].value_counts(dropna=False))

Before:
bp_meds_post
0.0    165
NaN     37
1.0     25
2.0     11
3.0      1
Name: count, dtype: int64
lipid_meds_post
0.0    175
NaN     36
1.0     19
2.0      9
Name: count, dtype: int64
dm_meds_post
0.0    161
NaN     37
2.0     25
1.0     16
Name: count, dtype: int64
After:
bp_meds_post
0.0    165
NaN     37
1.0     37
Name: count, dtype: int64
lipid_meds_post
0.0    175
NaN     36
1.0     28
Name: count, dtype: int64
dm_meds_post
0.0    161
1.0     41
NaN     37
Name: count, dtype: int64


## Create at_risk binary response variable column
If all are NAN, outputs NAN. Otherwise, outputs 1 if any of the 3 columns are 1.

In [236]:
df['at_risk'] = df[['bp_meds_post','lipid_meds_post','dm_meds_post']].max(axis=1)

print(df['at_risk'].value_counts(dropna=False))

at_risk
0.0    124
1.0     79
NaN     36
Name: count, dtype: int64


## Standardize specific_nht_used
Eliminate case errors, spelling errors, standardize brand names, handle multiple NHTs.

In [237]:
print("Before:")
print(df['specific_nht_used'].value_counts(dropna=False))

# Normalize text
col = 'specific_nht_used'
df[col] = df[col].str.strip().str.lower()

# Mapping
mapping = {
    'abiraterone': 'Abiraterone',
    'abiaterone': 'Abiraterone',
    'zytiga': 'Abiraterone',
    'darolutamide': 'Darolutamide',
    'enzalutamide': 'Enzalutamide',
    'enzaltamide': 'Enzalutamide',
    'apalutamide': 'Apalutamide',
    'apalutamide to darolutamide': 'Multiple'
}

# Apply mapping
df[col] = df[col].replace(mapping)

# Check
print("After:")
print(df[col].value_counts(dropna=False))

Before:
specific_nht_used
Abiraterone                    97
Darolutamide                   85
Enzalutamide                   19
Apalutamide                    12
NaN                             6
darolutamide                    6
abiraterone                     5
enzalutamide                    3
apalutamide                     1
ENzalutamide                    1
Apalutamide to Darolutamide     1
Enzaltamide                     1
Abiaterone                      1
Zytiga                          1
Name: count, dtype: int64
After:
specific_nht_used
Abiraterone     104
Darolutamide     91
Enzalutamide     24
Apalutamide      13
NaN               6
Multiple          1
Name: count, dtype: int64


## Standardize adt_agent
Eliminate case errors, spelling errors, standardize brand names, handle multiple ADTs.

In [238]:
print("Before:")
print(df['adt_agent'].value_counts(dropna=False))

col = 'adt_agent'

# Normalize
df[col] = df[col].str.strip().str.lower()

# Mapping
mapping = {
    # Orgovyx
    'orgovyx': 'Orgovyx',
    'orogovyx': 'Orgovyx',

    # Lupron
    'lupron': 'Lupron',
    'lupron depot': 'Lupron',
    'leupron': 'Lupron',
    'lupon': 'Lupron',
    'lpron': 'Lupron',

    # Bicalutamide / Casodex
    'bicalutamide': 'Bicalutamide',
    'casodex': 'Bicalutamide',

    # Firmagon
    'firmagon': 'Firmagon',

    # Multi / transitions / combos
    'firmagon to lupron': 'Multiple',
    'lupron + casodex to lupron': 'Multiple',
    'lupron, orgovyx (d/c)': 'Multiple',
    'firmagon to orgovyx': 'Multiple',
    'orgovyx and nubeqa': 'Multiple',
    'lupron, firmagon': 'Multiple',
    'bicalutamide + lupron': 'Multiple',
    'bicalutamide, then lupron': 'Multiple',
    'casodex to lpron': 'Multiple',
    'bicalutamide to lupron to orgovyx': 'Multiple',
    'lupron + bicalutamide': 'Multiple',
    'lupron to orgovyx': 'Multiple',
    'bicalutamide to lupron': 'Multiple',
    'casodex to lupron': 'Multiple',
    'casodex to firmagon': 'Multiple'
}

# Apply mapping
df[col] = df[col].replace(mapping)

# Check
print("After:")
print(df[col].value_counts(dropna=False))

Before:
adt_agent
Orgovyx                              70
Lupron                               49
NaN                                  44
Bicalutamide                         24
Firmagon                             18
orgovyx                               5
bicalutamide                          3
Orogovyx                              3
Firmagon to Lupron                    2
Lupron + Casodex to Lupron            2
Lupron, Orgovyx (d/c)                 2
Firmagon to orgovyx                   1
Orgovyx and Nubeqa                    1
Lupron, Firmagon                      1
bicalutamide + Lupron                 1
Lupron Depot                          1
bicalutamide, then Lupron             1
Casodex to Lpron                      1
Leupron                               1
Bicalutamide to lupron to orgovyx     1
lupron + bicalutamide                 1
lupron to orgovyx                     1
Bicalutamide to lupron                1
Casodex to Lupron                     1
Casodex to Firmagon   

## Standardize other columns that use 0/1/2 instead of 0/1

In [239]:
# List of columns by type
# Columns where 1 = Yes, 2 = No
binary_1_yes = [
    'bb_prior', 'ace_arb_prior', 'hx_hld',
    'hx_high_tg', 'statin_prior', 'other_lipid_prior', 'lipid_panel_checked', 'hx_cad', 
    'hx_mi_stent', 'hx_chf', 'hx_arrhythmia', 'hx_carotid', 'hx_pad', 'hx_cva', 'hx_dm2',
    'glucose_over_200', 'asa_use', 'cards_prior',
    'cards_post', 'cards_referral', 'diet_counseling', 'exercise_counseling',
    'echo_ordered'
]

# Columns where 0 = No, 1+ = Yes
count_cols = [
    'hx_smoking', 'bp_meds_prior', 'bp_meds_post', 'dm_noninsulin',
    'dm_meds_post', 'lipid_meds_post'
]

# Columns where 1 = No, 2/3 = Yes / outside
flip_cols = ['has_pcp', 'has_pcp_duplicate']

# Columns where 1 = Yes, 2 = No, 0 = NAN
na_zero_cols = ['hx_htn', 'on_insulin', 'a1c_checked']

# --- Conversions ---

#print("Before:")
#for col in binary_1_yes + count_cols + flip_cols + na_zero_cols + ['family_hx_hd']:
    #print(f"{col}:\n{df[col].value_counts(dropna=False)}\n")

# Standard yes/no (1 = yes)
for col in binary_1_yes:
    df[col] = df[col].where(df[col].isna(), (df[col] == 1).astype(int))

# Count-style
df[count_cols] = df[count_cols].where(
    df[count_cols].isna(),
    (df[count_cols] != 0).astype(int)
)

# Flipped encoding
for col in flip_cols:
    df[col] = df[col].where(df[col].isna(), (df[col] != 1).astype(int))

# Zeros to NAN
df[na_zero_cols] = df[na_zero_cols].replace({
    1: 1,
    2: 0,
    0: np.nan
})

# Special case: Family history (3 → NaN)
df['family_hx_hd'] = df['family_hx_hd'].replace({
    1: 1,
    2: 0,
    3: np.nan,
    0: np.nan
})

# Check
#print("After:")
#for col in binary_1_yes + count_cols + flip_cols + na_zero_cols + ['family_hx_hd']:
    #print(f"{col}:\n{df[col].value_counts(dropna=False)}\n")

In [240]:
# Save cleaned dataframe
df.to_csv(CLEANED_FILE, index = False)
print(f"Cleaned dataset saved to: {CLEANED_FILE}")

Cleaned dataset saved to: ..\data\processed\cardio_onc_prostate_04cleaned.csv
